# CriticalGraphRAG — Modelado

**ITBA — Ciencia de Datos Aplicada**

Chatbot de análisis de conflictos sobre el dataset ACLED (Israel, 2023) usando Knowledge Graph + RAG.

**Contenido:**

1. Problema y datos.
2. Arquitectura.
3. Pipeline reproducible.
4. Demostración en vivo — API deployada.


In [19]:
import requests
from IPython.display import display, Markdown

API_URL = "https://critigal-graph-rag.onrender.com/chat"

def ask(question: str, timeout: int = 120) -> str:
    resp = requests.post(API_URL, json={"message": question}, timeout=timeout)
    resp.raise_for_status()
    return resp.json()["response"]

def show(question: str):
    display(Markdown(f"**Pregunta:** {question}"))
    display(Markdown(f"**Respuesta:** {ask(question)}"))
    print()


## 1. Problema y datos

Sistema de Q&A en lenguaje natural sobre eventos del conflicto Israel/Palestina, combinando recuperación estructurada (Cypher sobre Knowledge Graph) y semántica (embeddings vectoriales).

**Dataset**: ACLED — Armed Conflict Location & Event Data. Filtro aplicado (`config/dataset_filter.yaml`):
- `country: ['Israel']`
- `year:    [2023]`
- Resultado: **4259 eventos** en los 6 distritos administrativos de Israel.

EDA detallado en `notebooks/02_eda_acled_israel.ipynb`.


## 2. Arquitectura

RAG sobre Knowledge Graph — no RAG plano sobre texto:

- ACLED es naturalmente relacional (Event ↔ Actor ↔ Location ↔ EventType ↔ Source). El KG preserva esas relaciones y habilita preguntas multi-hop.
- Las preguntas factuales (conteos, agregaciones, rankings) requieren Cypher exacto — el índice vectorial solo no las resuelve.
- El embedding semántico se reserva para `Event.notes`, evitando inflar el índice con texto estructurado.

**Flujo**:

```
Pregunta del usuario
        │
        ▼
   Agente LLM (Gemini 2.5 Flash, LangGraph ReAct)
        │
   ┌────┴────────────────┬──────────────────┐
   ▼                    ▼                  ▼
cypher_query    similarity_search    text2cypher
(catálogo)      (vectorial)          (LLM→Cypher)
        │                   │               │
        └───────────────────┴───────────────┘
                            ▼
                 Neo4j Knowledge Graph
```

**Tools:**

| Tool | Qué hace | Cuándo se usa |
|------|----------|---------------|
| `cypher_query` | Ejecuta una query predefinida de `config/cypher_library.yaml`; el planner elige `query_id` y parámetros. | Preguntas factuales / agregación / ranking |
| `similarity_search` | Embebe la pregunta y recupera top-k Events por similitud coseno sobre `Event.notes`. | Preguntas descriptivas o de contexto |
| `text2cypher` | El LLM genera Cypher dinámico a partir del schema; valida con `EXPLAIN` y reintenta si falla. | Fallback cuando ninguna query del catálogo aplica |


## 3. Pipeline reproducible

6 scripts independientes; cada uno persiste su salida en disco para reiniciar desde cualquier checkpoint.

```bash
poetry run python scripts/01_download.py                       # CSV crudo de ACLED
poetry run python scripts/02_explore.py                        # EDA
poetry run python scripts/03_build_graph.py                    # CSV → parquets nodos/rels
poetry run python scripts/04_embeddings.py                     # Event.notes → vectores (Gemini)
poetry run python scripts/05_load_neo4j.py --mode destructive  # carga Neo4j
poetry run python scripts/06_evaluate.py                       # evaluación
```

Salidas intermedias: `data/graph/` (parquets) y `data/event_embeddings.parquet`. El grafo se reconstruye en cualquier instancia Neo4j ejecutando solo el script `05`.


### Estado del Knowledge Graph (Neo4j)

| Métrica | Valor |
|---------|-------|
| Total nodos | 5 134 |
| Total relaciones | 36 329 |
| Vector index | `event_notes_embedding` (cosine, dim=1536) |

**Nodos por label:**

| Label | Count |
|-------|-------|
| Event | 4 259 |
| Location | 418 |
| Source | 272 |
| Actor | 139 |
| EventType | 25 |
| Month | 12 |
| ActorType | 6 |
| DisorderType | 3 |

**Relaciones por tipo:**

| Tipo | Count |
|------|-------|
| REPORTED_BY | 9 748 |
| INVOLVED_IN | 9 482 |
| OF_DISORDER | 4 263 |
| OF_TYPE | 4 259 |
| IN_MONTH | 4 259 |
| AT_LOCATION | 4 259 |
| HAS_TYPE | 40 |
| SUBTYPE_OF | 19 |


## 4. Demostración en vivo — API deployada

Las siguientes celdas consultan la API en producción: `https://critigal-graph-rag.onrender.com/chat`.
Cada par de preguntas está diseñado para ejercitar una tool específica del agente:

| Tool | Cuándo se activa |
|------|------------------|
| `cypher_query` | Preguntas factuales con query predefinida en el catálogo |
| `similarity_search` | Preguntas descriptivas que requieren búsqueda semántica sobre `Event.notes` |
| `text2cypher` | Preguntas sin query predefinida — el LLM genera Cypher dinámico |


### 4.1 Tool: `cypher_query`

Ejecuta queries Cypher predefinidas del catálogo (`config/cypher_library.yaml`). 
El planner elige el `query_id` y completa los parámetros.


In [20]:
# Dispara: count_events_by_country_month (country='Israel', month_id='2023-10')
show("¿Cuántos eventos registró ACLED en Israel durante octubre de 2023?")


**Pregunta:** ¿Cuántos eventos registró ACLED en Israel durante octubre de 2023?

**Respuesta:** ACLED registró 768 eventos en Israel durante octubre de 2023.

In [21]:
# Dispara: sum_fatalities_by_country_year (country='Israel', year=2023)
show("¿Cuántas fatalidades totales se registraron en Israel durante 2023?")


**Pregunta:** ¿Cuántas fatalidades totales se registraron en Israel durante 2023?

**Respuesta:** Se registraron un total de 1773 fatalidades en Israel durante 2023.

### 4.2 Tool: `similarity_search`

Embebe la pregunta y recupera los top-k `Event` más similares usando el índice vectorial 
sobre `Event.notes` (cosine similarity). Ideal para preguntas descriptivas o de contexto.


In [22]:
# Pregunta descriptiva → el agente no puede responderla con un conteo exacto;
# embedea la pregunta y recupera eventos con notas similares a "ataques con cohetes".
show("¿Cómo se describen los ataques con cohetes registrados en Israel en 2023?")


**Pregunta:** ¿Cómo se describen los ataques con cohetes registrados en Israel en 2023?

**Respuesta:** Los ataques con cohetes registrados en Israel en 2023 se describen de la siguiente manera:

*   **Perpetradores y Origen:** Fueron lanzados por el PIJ (Yihad Islámica Palestina) y militantes de Hamas desde Gaza, y por facciones palestinas (presuntamente operativos de Hamas) desde el Líbano.
*   **Fechas Destacadas:** Se registraron ataques significativos en abril (6 de abril), mayo (10-13 de mayo) y octubre (7 de octubre).
*   **Zonas Afectadas:** Los cohetes impactaron o causaron alertas en localidades del centro (Bat Yam, Holon, Ramla, Rishon LeZion), sur (Niram, Karmiyya, Kefar Azza, Nirim, Ofaqim, Kerem Shalom, Kissufim) y norte de Israel (Hanita, Shlomi).
*   **Impacto y Consecuencias:**
    *   A menudo, los cohetes aterrizaron en áreas abiertas o fueron interceptados por la Cúpula de Hierro, sin causar heridos ni impactos directos.
    *   Se reportaron daños a la propiedad, incluyendo un impacto directo en una casa en Niram, el techo de una guardería en Nirim, y el inicio de incendios en Karmiyya y Kefar Azza.
    *   Hubo heridos leves por metralla y personas lesionadas al correr a refugios, especialmente en los ataques de abril.
    *   Se activaron sirenas de advertencia en múltiples localidades.
*   **Volumen de Cohetes:**
    *   El 6 de abril, se dispararon 34 cohetes desde el Líbano.
    *   Entre el 10 y el 13 de mayo, se lanzaron aproximadamente 1,469 proyectiles, con un pico de alrededor de 507 el 10 de mayo y 373 el 11 de mayo.
    *   El 7 de octubre, militantes de Hamas dispararon alrededor de 3,000 cohetes desde Gaza.

In [23]:
# Contexto geográfico descriptivo → ninguna query del catálogo filtra por 'frontera con Líbano';
# la búsqueda vectorial recupera eventos cuyas notas mencionan incidentes fronterizos.
show("Describí incidentes que ocurrieron en zonas fronterizas con Líbano")


**Pregunta:** Describí incidentes que ocurrieron en zonas fronterizas con Líbano

**Respuesta:** Se registraron varios incidentes en las zonas fronterizas con Líbano en 2023:

*   **21 de marzo:** Dos soldados israelíes resultaron heridos al chocar con una mina terrestre cerca de la frontera con Líbano, presumiblemente cerca de Metulla.
*   **13 de agosto:** Un militante de Hezbollah lanzó un cóctel molotov desde el lado libanés hacia fuerzas militares israelíes cerca de Metulla, quienes respondieron con disparos.
*   **1 de octubre:** Manifestantes libaneses lanzaron piedras sobre la frontera cerca de Metulla. Las fuerzas militares israelíes respondieron con granadas de humo.
*   **5 de octubre:** Fuerzas militares libanesas retiraron una estructura que habían construido en el lado israelí de la frontera cerca de Shetula, presuntamente destinada a Hezbollah.
*   **7 de octubre:** Fuerzas israelíes abrieron fuego contra militantes de Hezbollah en motocicletas que intentaron cruzar la frontera cerca de Metulla.
*   **18 de octubre:** Hombres armados de Hezbollah abrieron fuego con armas pequeñas hacia los cuarteles de Zarit. Las fuerzas israelíes respondieron con fuego de artillería.
*   **5 de diciembre:** Una "aeronave hostil" cruzó desde Líbano a territorio israelí sobre Margaliyyot, probablemente lanzada por Hezbollah.
*   **10 de diciembre:** Fuerzas militares israelíes interceptaron un "objetivo aéreo" que cruzó desde Líbano en la región de Galilea occidental, sobre Klil, probablemente lanzado por Hezbollah.
*   **12 de diciembre:** Fuerzas militares israelíes interceptaron un "objetivo aéreo sospechoso" (probablemente un dron armado) que cruzó desde Líbano hacia el norte de Israel (Shetula), probablemente lanzado por Hezbollah.
*   **24 de diciembre:** Fuerzas militares israelíes interceptaron un objetivo aéreo sospechoso que cruzó desde Líbano sobre Dovev, probablemente lanzado por Hezbollah.

### 4.3 Tool: `text2cypher`

Fallback cuando ninguna query predefinida cubre la pregunta. 
El LLM genera Cypher dinámico a partir del schema del grafo, lo valida con `EXPLAIN` 
y lo ejecuta. Incluye lógica de retry con corrección de errores.


In [27]:
# Breakdown tipo×región×mes: no existe query en el catálogo para esta combinación.
# El agente delega a text2cypher para generar el Cypher dinámico.
show("¿Cuántos eventos de tipo 'Battles' (primario) se registraron en Israel en 2023?")


**Pregunta:** ¿Cuántos eventos de tipo 'Battles' (primario) se registraron en Israel en 2023?

**Respuesta:** Se registraron 167 eventos de tipo 'Battles' en Israel en 2023.

In [25]:
# Agregación por día exacto: el catálogo tiene sum_fatalities_by_country_date (fecha fija),
# pero no tiene 'top-1 día con más fatalidades a nivel global' → text2cypher.
show("¿Cuál fue el día con más fatalidades registradas en todo 2023?")


**Pregunta:** ¿Cuál fue el día con más fatalidades registradas en todo 2023?

**Respuesta:** El día con más fatalidades registradas en todo 2023 fue el 7 de octubre de 2023, con un total de 1569 fatalidades.